# Blender 4.0.2 saved-project scene integration (CPU only)

Runs the bounded ProjectStore-to-`.blend` reopen and rigid-body motion check. No video render or benchmark is performed.


In [ ]:
from pathlib import Path
import base64
import json
import runpy
import sys
import traceback
import zipfile

BUNDLE_B64 = "..."
output = Path("/kaggle/working/ai-engine-scene-integration")
output.mkdir(parents=True, exist_ok=True)
wrapper_report = output / "kaggle-upload-wrapper-report.json"
report = {"status": "FAILED", "errors": []}
try:
    bundle_file = output / "ai_engine_blender_scene_integration_bundle.zip"
    bundle_file.write_bytes(base64.b64decode(BUNDLE_B64, validate=True))
    bundle_root = output / "uploaded-source"
    bundle_root.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(bundle_file) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Embedded source bundle failed CRC validation.")
        for member in archive.infolist():
            destination = (bundle_root / member.filename).resolve()
            destination.relative_to(bundle_root.resolve())
        archive.extractall(bundle_root)
    sys.path.insert(0, str(bundle_root / "src"))
    result = runpy.run_path(str(bundle_root / "run_scene_integration_kaggle.py"), run_name="__main__")
    report["status"] = "PASSED" if result.get("RESULT") == 0 else "FAILED"
    report["scene_report"] = "/kaggle/working/ai-engine-scene-integration"
    report["embedded_bundle_sha256"] = "6d9491c39632d93e9659e6828dc1af46a9001d19ba2cf81833526de2d315a9df"
    if report["status"] != "PASSED":
        raise RuntimeError("The saved-project Blender integration test returned failure.")
except Exception as exc:
    report["errors"].append(f"{type(exc).__name__}: {exc}")
    report["traceback"] = traceback.format_exc()
finally:
    wrapper_report.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
if report["status"] != "PASSED":
    raise RuntimeError("Kaggle scene integration did not pass; see wrapper report.")
